## **Perhitungan Naïve Bayes pada Dataset Diabetes**

TAHAP 1: MEMBACA DATA

In [2]:
from google.colab import drive
import pandas as pd
import numpy as np

url_id = '1aDEAjdyTd8qtB-AMRmAYXzxU08jNPcY2'
file_path = f'https://drive.google.com/uc?id={url_id}'

try:
    df = pd.read_csv(file_path)
    print("\n✅ Data berhasil di-load dari Drive!")
except:
    print("\n❌ ERROR: File tidak ditemukan. Cek link Drive.")


✅ Data berhasil di-load dari Drive!


TAHAP 2: MENGHITUNG PRIOR P(C)

In [3]:
target_col = 'Outcome'
kelas_target = sorted(df[target_col].unique())
total_data = len(df)

prior_probs = {}

for kelas in kelas_target:
    jumlah_kelas = len(df[df[target_col] == kelas])
    prior_probs[kelas] = jumlah_kelas / total_data
    print(f"P(Outcome={kelas}) = {jumlah_kelas} / {total_data} = {prior_probs[kelas]:.4f}")

P(Outcome=0) = 500 / 768 = 0.6510
P(Outcome=1) = 268 / 768 = 0.3490


TAHAP 3: MENGHITUNG MEAN DAN VARIANCE

In [4]:
fitur_cols = [col for col in df.columns if col != target_col]

mean_dict = {0: {}, 1: {}}
var_dict = {0: {}, 1: {}}

print(f"{'Fitur':<25} | {'Kelas':<6} | {'Mean (μ)':<10} | {'Variance (σ²)':<10}")
print("-" * 65)

for fitur in fitur_cols:
    for kelas in kelas_target:
        data_fitur = df[df[target_col] == kelas][fitur]

        m = data_fitur.mean()
        v = data_fitur.var(ddof=1)

        mean_dict[kelas][fitur] = m
        var_dict[kelas][fitur] = v

        print(f"{fitur:<25} | {kelas:<6} | {m:<10.4f} | {v:<10.4f}")

Fitur                     | Kelas  | Mean (μ)   | Variance (σ²)
-----------------------------------------------------------------
Pregnancies               | 0      | 3.2980     | 9.1034    
Pregnancies               | 1      | 4.8657     | 13.9969   
Glucose                   | 0      | 109.9800   | 683.3623  
Glucose                   | 1      | 141.2575   | 1020.1395 
BloodPressure             | 0      | 68.1840    | 326.2747  
BloodPressure             | 1      | 70.8246    | 461.8980  
SkinThickness             | 0      | 19.6640    | 221.7105  
SkinThickness             | 1      | 22.1642    | 312.5722  
Insulin                   | 0      | 68.7920    | 9774.3454 
Insulin                   | 1      | 100.3358   | 19234.6733
BMI                       | 0      | 30.3042    | 59.1339   
BMI                       | 1      | 35.1425    | 52.7507   
DiabetesPedigreeFunction  | 0      | 0.4297     | 0.0895    
DiabetesPedigreeFunction  | 1      | 0.5505     | 0.1386    
Age             

TAHAP 4: DATA PASIEN BARU, FUNGSI GAUSSIAN DAN TAHAP 5: MENGHITUNG LIKELIHOOD

In [ ]:
pasien_baru = {
    'Pregnancies': 2,
    'Glucose': 130,
    'BloodPressure': 70,
    'SkinThickness': 25,
    'Insulin': 100,
    'BMI': 28.5,
    'DiabetesPedigreeFunction': 0.45,
    'Age': 33
}

def hitung_gaussian(x, mean, var):
    depan = 1 / np.sqrt(2 * np.pi * var)
    eksponen = np.exp(-((x - mean)**2) / (2 * var))
    return depan * eksponen

likelihoods = {0: {}, 1: {}}

for kelas in kelas_target:
    print(f"\n--- OUTCOME {kelas} ---")

    for fitur in fitur_cols:
        x_i = pasien_baru[fitur]
        mu = mean_dict[kelas][fitur]
        sigma2 = var_dict[kelas][fitur]

        prob = hitung_gaussian(x_i, mu, sigma2)

        likelihoods[kelas][fitur] = prob

        print(f"P({fitur}={x_i}|{kelas}) = {prob:.6e}")


--- OUTCOME 0 ---
P(Pregnancies=2|0) = 1.205369e-01
P(Glucose=130|0) = 1.138217e-02
P(BloodPressure=70|0) = 2.197473e-02
P(SkinThickness=25|0) = 2.512639e-02
P(Insulin=100|0) = 3.839098e-03
P(BMI=28.5|0) = 5.047062e-02
P(DiabetesPedigreeFunction=0.45|0) = 1.330816e+00
P(Age=33|0) = 3.378320e-02

--- OUTCOME 1 ---
P(Pregnancies=2|1) = 7.952297e-02
P(Glucose=130|1) = 1.173828e-02
P(BloodPressure=70|1) = 1.854887e-02
P(SkinThickness=25|1) = 2.227656e-02
P(Insulin=100|1) = 2.876513e-03
P(BMI=28.5|1) = 3.615457e-02
P(DiabetesPedigreeFunction=0.45|1) = 1.033082e+00
P(Age=33|1) = 3.395584e-02


TAHAP 6: MENGHITUNG POSTERIOR (SKOR AKHIR)

In [ ]:
print("Rumus: P(C|X) = P(C) * P(X1|C) * ... * P(Xn|C)\n")

skor_akhir = {}

for kelas in kelas_target:

    skor = prior_probs[kelas]

    print(f"Skor Outcome {kelas} = {prior_probs[kelas]:.4f}", end="")

    for fitur in fitur_cols:
        skor *= likelihoods[kelas][fitur]
        print(f" * {likelihoods[kelas][fitur]:.2e}", end="")

    skor_akhir[kelas] = skor

    print(f"\nTotal Skor Outcome {kelas} = {skor:.10e}\n")

Rumus: P(C|X) = P(C) * P(X1|C) * ... * P(Xn|C)

Skor Outcome 0 = 0.6510 * 1.21e-01 * 1.14e-02 * 2.20e-02 * 2.51e-02 * 3.84e-03 * 5.05e-02 * 1.33e+00 * 3.38e-02
Total Skor Outcome 0 = 4.2962921123e-12

Skor Outcome 1 = 0.3490 * 7.95e-02 * 1.17e-02 * 1.85e-02 * 2.23e-02 * 2.88e-03 * 3.62e-02 * 1.03e+00 * 3.40e-02
Total Skor Outcome 1 = 4.9103765606e-13



TAHAP 7: NORMALISASI PROBABILITAS

In [ ]:
total_skor = skor_akhir[0] + skor_akhir[1]

posterior_prob = {
    0: skor_akhir[0] / total_skor,
    1: skor_akhir[1] / total_skor
}

print(f"P(Outcome=0 | X) = {posterior_prob[0]:.6f}")
print(f"P(Outcome=1 | X) = {posterior_prob[1]:.6f}")

P(Outcome=0 | X) = 0.897430
P(Outcome=1 | X) = 0.102570


TAHAP 8: PENENTUAN HASIL KLASIFIKASI

In [ ]:
if posterior_prob[1] > posterior_prob[0]:

    print(f"Probabilitas Diabetes = {posterior_prob[1]:.6f}")
    print("Prediksi: KELAS 1 (TERKENA DIABETES)")

else:

    print(f"Probabilitas Tidak Diabetes = {posterior_prob[0]:.6f}")
    print("Prediksi: KELAS 0 (TIDAK DIABETES)")

Probabilitas Tidak Diabetes = 0.897430
Prediksi: KELAS 0 (TIDAK DIABETES)
